# M2 - Agentic AI - 使用 Reflection 改进 SQL 生成

## 1. 简介
### 1.1 实验概述

在本实验中，你将探索 **Reflection（反思）模式** 如何改进一个智能体工作流：把自然语言问题转换成 SQL 数据库查询。你会看到智能体如何发现自身输出中的问题、进行修正，并在给出最终答案前不断改进结果。

### 🎯 1.2 学习目标

你将练习把 Reflection 模式应用到 SQL 生成工作流中，使智能体能够：

* 审查自己的中间结果（如草稿 SQL 或工具输出）。
* 识别错误或遗漏。
* 检查自己的回复与工具使用是否正确。
* 在提交最终答案前优化输出。

## 2. 环境准备：初始化环境与客户端

在这一步，你将准备工作区，以便立即开始编码。具体包括：

1. **导入核心 Python 库**

   * `json`：处理结构化数据。
   * `pandas`：处理表格数据。
   * `dotenv`：加载环境变量（如 API Key）。

2. **加载环境变量**
   确保工作区已正确配置所需的密钥和参数。

3. **导入 `utils.py` 模块**
   该文件包含用于格式化输出、支持后续工作流步骤的辅助函数。

**提示：** 若想查看 `utils.py` 的内容，可在顶部菜单选择 **File > Open**。

In [1]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

### 2.1 初始化智谱 LLM 客户端

本实验使用智谱 AI 的 OpenAI 兼容接口调用大语言模型。下面初始化 `ZhipuClient`，它会读取 `.env` 中的 `ZHIPU_API_KEY` 和模型配置，提供统一的对话调用方式。

In [2]:
from config import get_settings
from llm import ZhipuClient

settings = get_settings()
client = ZhipuClient(settings)
MODEL_GENERATION = settings.zhipu_model_generation
MODEL_EVALUATION = settings.zhipu_model_evaluation


### 2.2 创建数据库

在这一步，你将创建一个本地 SQLite 数据库 **`products.db`**。
数据库会自动填充随机生成的商品交易数据。

你将在后续实验中用它练习和测试 SQL 查询。

In [3]:
utils.create_transactions_db()

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


执行下面的单元格可以查看表结构。

In [4]:
utils.print_html(utils.get_schema('products.db'))

在本表中，每一行代表一个**事件**（insert、restock、sale 或 price_update）。库存、销售额、价格趋势等分析指标都从这些事件聚合得出。

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">  
<strong>🔎 表结构概览：</strong><br><br>  
• <code>id</code> → 唯一事件 ID（自增）。<br>  
• <code>product_id</code>、<code>product_name</code>、<code>brand</code>、<code>category</code>、<code>color</code> → 标识商品。<br>  
• <code>action</code> → 事件类型（<em>insert</em>、<em>restock</em>、<em>sale</em>、<em>price_update</em>）。<br>  
• <code>qty_delta</code> → 库存变化（入库/补货为正，销售为负，改价时为 0）。<br>  
• <code>unit_price</code> → 该时刻价格（补货时为 NULL）。<br>  
• <code>notes</code> → 事件说明（可选）。<br>  
• <code>ts</code> → 事件记录时间戳。<br>  
</div>  

> 借助该表结构，你可以通过聚合事件历史，随时还原**当前状态**（库存、最新价格、总销售额等）。

## 3. 构建 SQL 生成器

### 3.1 使用 LLM 查询数据库

在这一步，你将使用一个函数，把自然语言问题转换成 SQL 查询。

你提供问题和数据库表结构作为输入，LLM 会生成能回答该问题的 SQL。

这样，**你**只需专注于提问，而由模型负责编写查询语句。

In [5]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
你是一名 SQL 助手。根据表结构和用户问题，为 SQLite 编写 SQL 查询。

表结构：
{schema}

用户问题：
{question}

只返回 SQL，不要其他说明。
"""
    sql = client.chat(
        prompt,
        model=model,
        temperature=settings.zhipu_temperature_generation,
    )
    return sql.removeprefix("```sql").removesuffix("```").strip()


运行下面的单元格，查看 **`generate_sql`** 如何根据自然语言问题，为 `transactions` 表生成 SQL 的**第一版（V1）**。
只需提供**问题**、**表结构**和**模型名称**，函数就会返回初始查询草稿。

In [6]:
# generate_sql 使用示例

schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

question = "哪种颜色的商品总销售额最高？"

utils.print_html(question, title="用户问题")

sql_V1 = generate_sql(question, schema, model=MODEL_GENERATION)

utils.print_html(sql_V1, title="SQL 查询 V1")


#### 3.1.1 执行查询

现在你将执行 SQL 查询的**第一版（V1）**并查看结果。这一步很重要，因为它能帮你验证 LLM 生成的查询是否真正取回了你期望的数据。

- **`utils.execute_sql(...)`**：在 `products.db` 上运行生成的 SQL（V1），并以 pandas DataFrame 返回结果，便于后续检查和分析。

- **`utils.print_html(...)`**：把 DataFrame 渲染为格式化的 HTML 表格，让输出更易读，也更容易判断结果是否符合用户问题。

In [7]:
# 在 products.db 上执行生成的 SQL（V1）
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# 以 HTML 表格展示结果，便于阅读
utils.print_html(df_sql_V1, title="SQL 查询 V1 的输出 - ❌ 未能完整回答问题")


color,total_sales
blue,-150511.18


<div style="border:1px solid #ef4444; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ <strong>V1 输出问题：</strong> 查询结果并未完全回答问题，因为总销售额为<em>负数</em>（如 `-190571.46`）。  
<br><br>
在本数据集中，<code>qty_delta</code> 在**销售**时记录为**负数**（库存减少），在**退货或补货**时记录为**正数**（库存增加）。若查询使用 <code>SUM(qty_delta)</code>，跨交易求和时负数会占主导，从而得到负的总销售额。  
<br>
这意味着 SQL 在语法上有效，但在语义上不正确——总销售额不应为负。  
<br>
➡️ 这正是<strong>需要 Reflection</strong> 的原因：模型必须改进逻辑（例如在销售计算时对 <code>qty_delta</code> 乘以 <code>-1</code>，或使用 <code>ABS()</code>），使最终查询反映真实的销售总额。  

</div>

### 3.2 使用 Reflection 改进 SQL 查询

在本节中，你将学习如何**通过 Reflection 改进 SQL 查询**。

首先，LLM 可以仅根据**问题**和**表结构**审查 SQL 文本本身，并在需要时提出改进建议。
随后，LLM 还可以结合**实际查询执行结果**，发现负值总额、缺少过滤条件、分组错误等细微问题。

这两种方式共同说明：Reflection 如何让 SQL 工作流更**可靠、更准确**——先检查逻辑，再用真实数据验证。

#### 3.2.1 第一次尝试：仅基于 SQL 文本反思

在这个函数中，你让 LLM **审查**一条 SQL 查询，对照原始问题和表结构（如 3.1 节所定义）。模型会反思该查询是否完整回答问题；若不能，则提出改进版本。

* **输入**：

  * 用户的**问题**
  * **原始 SQL 查询**
  * **表结构**

* **输出**：

  * **feedback** → 简短评价（如「语法正确但缺少日期过滤」）
  * **refined_sql** → 最终 SQL（若已正确则不变，否则返回改进版）

该函数**不会执行 SQL**，只检查查询逻辑，并在与意图不完全匹配时给出修改建议。

In [8]:
from llm import parse_json_object

def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """仅基于 SQL 文本进行反思，返回 (feedback, refined_sql)。"""
    prompt = f"""
你是一名 SQL 审查与改进助手。

用户问题：
{question}

原始 SQL：
{sql_query}

表结构：
{schema}

步骤 1：简要评价该 SQL 的输出是否能完整回答用户问题。
步骤 2：如需改进，请提供适用于 SQLite 的改进版 SQL；若已正确则原样返回。

请严格返回 JSON，包含两个字段：
{{
  "feedback": "<1-3 句说明问题或确认正确>",
  "refined_sql": "<最终要执行的 SQL>"
}}
"""
    content = client.chat(
        prompt,
        model=model,
        temperature=settings.zhipu_temperature_evaluation,
    )

    try:
        obj = parse_json_object(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql


Run the following cell to produce the **refined SQL query (V2)**. This step will:

* Display the **initial SQL query (V1)** generated in Section 3.1 for the question *“Which color of product has the highest total sales?”*
* Show the model’s **feedback** and its **refined SQL proposal (V2)**.
* Execute the **original SQL (V1)** against the database and present its **actual output**, so you can see why refinement was needed.

In [9]:
# 示例：将 SQL 从 V1 反思改进为 V2（仅基于 SQL 文本）

feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,
    schema=schema,
    model=MODEL_EVALUATION,
)

utils.print_html(question, title="用户问题")
utils.print_html(sql_V1, title="生成的 SQL 查询（V1）")

df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_V1, title="V1 的 SQL 输出 - ❌ 未能完整回答问题")

utils.print_html(feedback, title="对 V1 的反馈")
utils.print_html(sql_V2, title="改进后的 SQL 查询（V2）")

df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="V2 的 SQL 输出 - ❌ 未能完整回答问题")


color,total_sales
blue,-150511.18


color,total_sales
blue,-150511.18
green,-183924.06
red,-202085.84
black,-224133.27
white,-289877.73


<div style="border:1px solid #fecaca; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ **如你所见……**  
尽管生成的 SQL 看起来正确，反馈也认可了它，但**实际查询结果**仍显示 `total_sales` 为负。  

如前所述，这是因为 SQL 直接计算 `qty_delta * unit_price`，而没有考虑销售事件的 **数量为负**（`qty_delta < 0`）。正负号是一个细微的语义问题，仅审查 SQL 文本往往难以发现。  

👉 因此，智能体还必须对**执行输出**进行反思——以发现符号错误、缺少过滤条件或聚合方式不当等问题。来自查询结果的外部反馈，能让改进过程建立在真实数据之上，而不仅是 SQL 结构。  

</div>

#### 3.2.2 最终方案：结合外部反馈改进 SQL

与上一步仅**审查 SQL 文本**不同，现在你将把**实际查询执行结果**作为外部反馈提供给模型。
该反馈来自在数据库上运行 SQL——与课程视频中的示例一样——让 LLM 能根据真实输出判断查询是否真正回答了问题。

在本步中，你将获得：

* 基于**真实执行输出**的简短评价。  
* 具体改进建议（如缺少过滤、分组问题、符号错误等）。  
* 一条更贴合原始问题的改进版 SQL。  

In [10]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """结合 SQL 执行结果进行反思，返回 (feedback, refined_sql)。"""
    prompt = f"""
你是一名 SQL 审查与改进助手。

用户问题：
{question}

原始 SQL：
{sql_query}

SQL 执行结果：
{df_feedback.to_markdown(index=False)}

表结构：
{schema}

步骤 1：简要评价 SQL 输出是否回答了用户问题。
步骤 2：若可改进，请提供改进版 SQL；若已正确则原样返回。

请严格返回 JSON，包含两个字段：
- "feedback": 简要评价与建议
- "refined_sql": 最终要执行的 SQL
"""

    content = client.chat(
        prompt,
        model=model,
        temperature=settings.zhipu_temperature_evaluation,
    )

    try:
        obj = parse_json_object(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql


运行下面的单元格，查看**外部反馈**如何帮助改进 SQL。

本示例将：

* 展示根据问题生成的**原始 SQL（V1）**。
* 显示 **V1 的输出**，说明初次尝试为何未能完整回答问题。
* 提供 LLM 基于该输出给出的**反馈**。
* 展示解决问题的**改进 SQL（V2）**。
* 执行 **V2** 并展示输出，验证其是否 ✅ 完整回答问题。

In [11]:
# 示例：结合外部反馈将 SQL 从 V1 改进为 V2

df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,
    df_feedback=df_sql_V1,
    schema=schema,
    model=MODEL_EVALUATION,
)

utils.print_html(question, title="用户问题")
utils.print_html(sql_V1, title="生成的 SQL 查询（V1）")
utils.print_html(df_sql_V1, title="V1 的 SQL 输出 - ❌ 未能完整回答问题")

utils.print_html(feedback, title="对 V1 的反馈")
utils.print_html(sql_V2, title="改进后的 SQL 查询（V2）")

df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="V2 的 SQL 输出（含外部反馈）- ✅ 完整回答问题")


color,total_sales
blue,-150511.18


color,total_sales
white,68437.36
black,51042.88
blue,40060.28
red,39989.39
green,30540.64


<div style="border:1px solid #bbf7d0; border-left:6px solid #22c55e; background:#dcfce7; border-radius:6px; padding:12px 14px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

✅ **成功！**  
在改进后的 SQL 中，通过对 `qty_delta` 使用 `ABS(qty_delta)`（或对销售事件使用 `-qty_delta`），修正了负值问题。  

现在输出为**正数且有意义**，能正确显示总销售额最高的颜色。这体现了 **Reflection + 外部反馈** 的重要性：  
- 仅看 SQL 文本似乎没问题。  
- 实际执行输出暴露了符号错误。  
- 改进后的查询修正了逻辑，得到正确答案。  

</div>

### 3.3 整合全流程 —— 构建数据库查询工作流

在这一步，**你**将使用一个函数，自动化「生成、执行、改进 SQL」的完整 LLM 工作流。

工作流包含以下关键步骤：

1. **提取数据库表结构**
2. 根据自然语言问题**生成初始 SQL（V1）**
3. **结合执行结果反思 V1**，必要时改进 SQL
4. **执行改进后的 SQL（V2）**，确保完整回答问题

最后，**你**将看到：

* 初始版与改进版 SQL
* 各自的执行输出
* LLM 对改进原因的说明

这能让你以更顺畅、更准确、更自动化的方式处理 SQL 查询。

In [12]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = MODEL_GENERATION,
    model_evaluation: str = MODEL_EVALUATION,
):
    """端到端 SQL 工作流：生成 V1 → 执行 → 反思 → 改进为 V2 → 再执行。"""
    schema = utils.get_schema(db_path)
    utils.print_html(schema, title="📘 步骤 1 — 提取数据库表结构")

    sql_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(sql_v1, title="🧠 步骤 2 — 生成 SQL（V1）")

    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(df_v1, title="🧪 步骤 3 — 执行 V1（SQL 输出）")

    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,
        schema=schema,
        model=model_evaluation,
    )
    utils.print_html(feedback, title="🧭 步骤 4 — 反思 V1（反馈）")
    utils.print_html(sql_v2, title="🔁 步骤 4 — 改进后的 SQL（V2）")

    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(df_v2, title="✅ 步骤 5 — 执行 V2（最终答案）")


### 3.4 运行 SQL 工作流

现在轮到**你**执行完整的 SQL 处理流水线。可以尝试以下智谱模型的不同组合，它们在能力与性能上各有特点：

* `glm-4-flash`（快速，适合生成）
* `glm-4-plus`（更强，适合反思）
* `glm-4-air`（均衡）
* `glm-4-long`（长上下文）

💡 在本工作流中，`glm-4-plus` 通常在自我反思任务上表现更好。

**注意：** 大语言模型具有随机性，每次运行结果可能略有不同。
建议你尝试不同模型组合，找到最适合**你**的配置。

In [13]:
run_sql_workflow(
    "products.db",
    "哪种颜色的商品总销售额最高？",
    model_generation=MODEL_GENERATION,
    model_evaluation=MODEL_EVALUATION,
)


color,total_sales
blue,-150511.18


color,total_sales


## 4. 总结

完成本实验后，**你**已经学会：

* 使用 LLM 将自然语言问题转换为 SQL 查询。  
* 应用 **Reflection 模式**（含/不含外部反馈）改进生成的 SQL。  
* 自动化完整的 SQL 工作流：从提取表结构到执行与改进查询。  
* 尝试不同 LLM 模型，比较性能与准确性。  

核心启示是：Reflection 让**你的**智能体更可靠——不再停留在第一次尝试，而是能够审查、改进，并交付更符合**你**意图的结果。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>  

你已完成**智能体 SQL 工作流（V1 → V2）**实验。  

在此过程中，**你**实践了 Reflection、执行与验证如何组合成可靠流水线。  
你也看到了**外部反馈**的重要性：有时 SQL 文本本身看起来正确，但实际执行输出会暴露隐藏问题（如负值总额、缺少过滤、分组错误等）。  

掌握这些技能后，你已能设计自己的**智能体流水线**——工作流能够：  
* 生成初始查询（V1）。  
* 结合执行反馈进行反思与改进（V2）。  
* 交付更安全、更透明、更易信赖的结果。 🌟  

</div>